# SAP BTP Integration and Deployment Packaging
Date: Friday, October 30, 2026
SAP BTP integration and deployment
Outcome: Wrap the governed Finance assistant in an HTTP API, test its controls, and generate an SAP BTP Cloud Foundry deployment package.

Today produces a deployment package—not a live deployment. SAP BTP Cloud Foundry supports Python 3.11 through the Python buildpack and uses manifest.yml to describe the application. SAP Help Portal

SAP’s current Python AI SDK can obtain its configuration from environment variables or VCAP_SERVICES, allowing credentials to remain outside source control. SAP Help Portal

# Secure BTP configuration

- Read BTP credentials from environment variables or service bindings.
- Never store client secrets or tokens in GitHub.
- Use the Cloud Foundry-provided PORT.
- Bind XSUAA and SAP AI Core services during actual deployment.
- Log request identifiers and outcomes, but avoid raw sensitive prompts.
Databricks similarly provides service logs, endpoint-health metrics and governed inference tables for operational monitoring and audit use cases.

# Install the local API dependencies

In [1]:
# DAY 24 — INSTALL API DEVELOPMENT DEPENDENCIES
#
# Purpose:
# Install the small set of packages required to create and locally
# test the deployment API.
#
# %pip installs into the active sap-ai-engineer notebook kernel.

%pip install -q "fastapi>=0.115,<1.0" "uvicorn>=0.30,<1.0" "httpx>=0.27,<1.0"

Note: you may need to restart the kernel to use updated packages.


# Locate the repository and review the deployment gate

In [20]:
# DAY 24 — CORRECT PROJECT ROOT

from pathlib import Path
import os
import sys


projects_area = Path(
    "/var/www/filebrowser/.projects"
)

repository_candidates = [
    Path.cwd().resolve(),
    *Path.cwd().resolve().parents,
]

if projects_area.exists():
    repository_candidates.extend(
        projects_area.glob(
            "*/sap-finance-ai-assistant"
        )
    )


PROJECT_ROOT = next(
    (
        path.resolve()
        for path in repository_candidates
        if path.exists()
        and (path / ".git").exists()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "The Git repository could not be located."
    )


os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(
        0,
        str(PROJECT_ROOT),
    )


required_files = [
    Path("app") / "btp_runtime.py",
    Path("app") / "btp_service.py",
    (
        Path("governance")
        / "day23_release_decision.json"
    ),
    (
        Path("artifacts")
        / "day23_evaluation_report.json"
    ),
    Path("manifest.yml"),
]


print("Correct project root:")
print(PROJECT_ROOT)

print("\nReadiness prerequisites:")

missing_files = []

for relative_path in required_files:
    if (PROJECT_ROOT / relative_path).exists():
        print("✅", relative_path)
    else:
        print("❌", relative_path)
        missing_files.append(str(relative_path))


if missing_files:
    raise FileNotFoundError(
        "Missing readiness prerequisites: "
        f"{missing_files}"
    )

Correct project root:
/var/www/filebrowser/.projects/b1f2f927-84af-459c-80e3-1fe60dbc8858/sap-finance-ai-assistant

Readiness prerequisites:
✅ app/btp_runtime.py
✅ app/btp_service.py
✅ governance/day23_release_decision.json
✅ artifacts/day23_evaluation_report.json
✅ manifest.yml


In [21]:
# DAY 24 — PROJECT AND GOVERNANCE SETUP
#
# Purpose:
# Locate the existing Git repository and read yesterday's governance
# decision without exposing or changing credentials.

from pathlib import Path
from datetime import datetime, timezone
import importlib.util
import json
import os
import sys


def locate_project_root(starting_folder: Path) -> Path:
    """
    Find the repository root from the current notebook location.
    """

    for possible_root in [
        starting_folder,
        *starting_folder.parents,
    ]:
        has_git = (
            possible_root / ".git"
        ).exists()

        has_project_folders = (
            (possible_root / "app").exists()
            and (possible_root / "data").exists()
        )

        if has_git or has_project_folders:
            return possible_root

    return starting_folder


PROJECT_ROOT = locate_project_root(
    Path.cwd().resolve()
)

required_folders = [
    "app",
    "artifacts",
    "audit",
    "deployment",
    "governance",
    "notes",
    "notebooks",
]

for folder_name in required_folders:
    (PROJECT_ROOT / folder_name).mkdir(
        parents=True,
        exist_ok=True,
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


def save_json(relative_path: str, payload) -> Path:
    """
    Save formatted JSON inside the project.
    """

    destination = PROJECT_ROOT / relative_path

    destination.write_text(
        json.dumps(
            payload,
            indent=2,
            ensure_ascii=False,
        )
        + "\n",
        encoding="utf-8",
    )

    return destination


day23_decision_path = (
    PROJECT_ROOT
    / "governance"
    / "day23_release_decision.json"
)

if day23_decision_path.exists():
    day23_decision = json.loads(
        day23_decision_path.read_text(
            encoding="utf-8"
        )
    )
else:
    # A safe default prevents an absent decision from accidentally
    # authorizing the service.
    day23_decision = {
        "automated_technical_gate": "UNKNOWN",
        "human_approval_status": "UNKNOWN",
        "deployment_authorized": False,
    }

print("Project root:")
print(PROJECT_ROOT)

print("\nDay 23 technical gate:")
print(
    day23_decision[
        "automated_technical_gate"
    ]
)

print("\nHuman approval:")
print(
    day23_decision[
        "human_approval_status"
    ]
)

print("\nDeployment authorized:")
print(
    day23_decision[
        "deployment_authorized"
    ]
)

Project root:
/var/www/filebrowser/.projects/b1f2f927-84af-459c-80e3-1fe60dbc8858/sap-finance-ai-assistant

Day 23 technical gate:
PASS

Human approval:
PENDING

Deployment authorized:
False


# Create the deployment API

In [22]:
# DAY 24 — CREATE THE BTP-COMPATIBLE API
#
# Purpose:
# Wrap the governed Finance assistant behavior in an HTTP API.
#
# The service provides:
# - /health for application health
# - /ready for technical-gate readiness
# - /configuration for safe configuration status
# - /ask for governed Finance requests
#
# This development version uses deterministic responses. A future
# version will replace that section with SAP Generative AI Hub calls.

service_source = '''
"""Deployment API for the governed SAP Finance assistant."""

from datetime import datetime, timezone
from pathlib import Path
from typing import Literal
from uuid import uuid4
import hashlib
import json
import os

from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field


SERVICE_NAME = "governed-sap-finance-assistant"
SERVICE_VERSION = "0.24.0"

PROJECT_ROOT = Path(__file__).resolve().parents[1]

DECISION_PATH = (
    PROJECT_ROOT
    / "governance"
    / "day23_release_decision.json"
)

AUDIT_PATH = (
    PROJECT_ROOT
    / "audit"
    / "day24_api_audit.jsonl"
)


ROLE_POLICY_ACCESS = {
    "finance_analyst": {
        "FIN-JE-001",
        "FIN-CLOSE-001",
    },
    "finance_manager": {
        "FIN-JE-001",
        "FIN-CLOSE-001",
    },
    "controller": {
        "FIN-JE-001",
        "FIN-CLOSE-001",
    },
    "auditor": {
        "FIN-JE-001",
        "FIN-CLOSE-001",
    },
}


class FinanceQuestion(BaseModel):
    """Validated incoming Finance-assistant request."""

    user_id: str = Field(
        min_length=1,
        max_length=100,
    )

    role: str = Field(
        min_length=1,
        max_length=50,
    )

    business_unit: str = Field(
        min_length=1,
        max_length=50,
    )

    question: str = Field(
        min_length=3,
        max_length=2000,
    )


class FinanceDecision(BaseModel):
    """Core governed output contract."""

    answer: str

    decision: Literal[
        "answer",
        "refuse",
        "escalate",
    ]

    requires_human_approval: bool

    citations: list[str]

    rationale: str


class FinanceApiResponse(BaseModel):
    """API envelope containing the governed decision."""

    request_id: str
    result: FinanceDecision
    audit_event_id: str


def read_governance_decision() -> dict:
    """
    Read the latest release-gate decision.

    Missing governance evidence always produces a safe,
    unauthorized default.
    """

    if not DECISION_PATH.exists():
        return {
            "automated_technical_gate": "UNKNOWN",
            "human_approval_status": "UNKNOWN",
            "deployment_authorized": False,
        }

    return json.loads(
        DECISION_PATH.read_text(
            encoding="utf-8"
        )
    )


def append_audit_event(event: dict) -> None:
    """
    Append one JSON event to the audit log.
    """

    AUDIT_PATH.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    with AUDIT_PATH.open(
        "a",
        encoding="utf-8",
    ) as audit_file:
        audit_file.write(
            json.dumps(event)
            + "\\n"
        )


def question_hash(question: str) -> str:
    """
    Create a traceable fingerprint without logging the raw question.
    """

    return hashlib.sha256(
        question.encode("utf-8")
    ).hexdigest()


def identify_policy(question: str):
    """
    Map the synthetic question to an approved policy.

    This is a deterministic deployment smoke-test implementation,
    not an LLM or production retrieval system.
    """

    normalized_question = question.lower()

    if (
        "journal" in normalized_question
        or "100,000" in normalized_question
        or "125,000" in normalized_question
    ):
        return {
            "policy_id": "FIN-JE-001",
            "answer": (
                "Journal entries above USD 100,000 require "
                "approval by a finance manager."
            ),
        }

    if (
        "period-end" in normalized_question
        or "adjustment" in normalized_question
        or "50,000" in normalized_question
    ):
        return {
            "policy_id": "FIN-CLOSE-001",
            "answer": (
                "Period-end adjustments above USD 50,000 "
                "require approval by a regional finance manager."
            ),
        }

    return None


app = FastAPI(
    title="Governed SAP Finance Assistant",
    version=SERVICE_VERSION,
)


@app.get("/health")
def health():
    """
    Confirm that the application process is operating.
    """

    return {
        "status": "healthy",
        "service": SERVICE_NAME,
        "version": SERVICE_VERSION,
    }


@app.get("/ready")
def ready():
    """
    Confirm that the automated technical gate passed.

    This does not claim that production deployment was authorized.
    """

    governance = read_governance_decision()

    if (
        governance.get(
            "automated_technical_gate"
        )
        != "PASS"
    ):
        raise HTTPException(
            status_code=503,
            detail={
                "status": "not_ready",
                "reason": (
                    "The automated technical gate has not passed."
                ),
            },
        )

    return {
        "status": "technical_ready",
        "technical_gate": "PASS",
        "human_approval_status": governance.get(
            "human_approval_status"
        ),
        "deployment_authorized": governance.get(
            "deployment_authorized",
            False,
        ),
    }


@app.get("/configuration")
def configuration():
    """
    Return only whether required SAP AI Core configuration exists.

    Credential values are never returned.
    """

    required_fields = [
        "AICORE_CLIENT_ID",
        "AICORE_CLIENT_SECRET",
        "AICORE_AUTH_URL",
        "AICORE_BASE_URL",
        "AICORE_RESOURCE_GROUP",
    ]

    field_status = {
        field_name: bool(
            os.getenv(field_name)
        )
        for field_name in required_fields
    }

    return {
        "sap_ai_core_configuration_complete": all(
            field_status.values()
        ),
        "credential_fields_present": field_status,
        "credential_values_exposed": False,
    }


@app.post(
    "/ask",
    response_model=FinanceApiResponse,
)
def ask_finance_assistant(
    request: FinanceQuestion,
):
    """
    Process a governed synthetic Finance request.
    """

    request_id = uuid4().hex
    audit_event_id = uuid4().hex

    governance = read_governance_decision()

    app_environment = os.getenv(
        "APP_ENV",
        "development",
    ).lower()

    # Production requests remain blocked until the governance
    # record explicitly authorizes deployment.
    if (
        app_environment == "production"
        and not governance.get(
            "deployment_authorized",
            False,
        )
    ):
        append_audit_event(
            {
                "audit_event_id": audit_event_id,
                "request_id": request_id,
                "timestamp_utc": datetime.now(
                    timezone.utc
                ).isoformat(),
                "event": "production_request_blocked",
                "question_sha256": question_hash(
                    request.question
                ),
                "role": request.role,
                "business_unit": request.business_unit,
            }
        )

        raise HTTPException(
            status_code=503,
            detail={
                "decision": "blocked",
                "reason": (
                    "Production deployment has not received "
                    "human authorization."
                ),
            },
        )

    normalized_role = request.role.lower()

    if normalized_role not in ROLE_POLICY_ACCESS:
        append_audit_event(
            {
                "audit_event_id": audit_event_id,
                "request_id": request_id,
                "timestamp_utc": datetime.now(
                    timezone.utc
                ).isoformat(),
                "event": "role_access_denied",
                "question_sha256": question_hash(
                    request.question
                ),
                "role": request.role,
                "business_unit": request.business_unit,
            }
        )

        raise HTTPException(
            status_code=403,
            detail={
                "decision": "refuse",
                "reason": "The role is not authorized.",
            },
        )

    policy = identify_policy(
        request.question
    )

    action_words = [
        "post",
        "approve",
        "reverse",
        "modify",
        "execute",
    ]

    normalized_question = (
        request.question.lower()
    )

    action_requested = any(
        action_word in normalized_question
        for action_word in action_words
    )

    permitted_policies = ROLE_POLICY_ACCESS[
        normalized_role
    ]

    if (
        policy is not None
        and policy["policy_id"]
        not in permitted_policies
    ):
        policy = None

    if action_requested:
        result = {
            "answer": (
                "I cannot execute the requested financial "
                "transaction. A human approver must review it."
            ),
            "decision": "escalate",
            "requires_human_approval": True,
            "citations": (
                [policy["policy_id"]]
                if policy is not None
                else []
            ),
            "rationale": (
                "Financial execution is outside the assistant's "
                "authority."
            ),
        }

    elif policy is not None:
        result = {
            "answer": policy["answer"],
            "decision": "answer",
            "requires_human_approval": False,
            "citations": [
                policy["policy_id"]
            ],
            "rationale": (
                "The answer is supported by an authorized "
                "synthetic Finance policy."
            ),
        }

    else:
        result = {
            "answer": (
                "I cannot answer because no authorized policy "
                "supports the request."
            ),
            "decision": "refuse",
            "requires_human_approval": False,
            "citations": [],
            "rationale": (
                "No authorized context was available."
            ),
        }

    append_audit_event(
        {
            "audit_event_id": audit_event_id,
            "request_id": request_id,
            "timestamp_utc": datetime.now(
                timezone.utc
            ).isoformat(),
            "event": "finance_request_completed",
            "question_sha256": question_hash(
                request.question
            ),
            "role": request.role,
            "business_unit": request.business_unit,
            "decision": result["decision"],
            "citations": result["citations"],
            "requires_human_approval": result[
                "requires_human_approval"
            ],
        }
    )

    return {
        "request_id": request_id,
        "result": result,
        "audit_event_id": audit_event_id,
    }
'''

service_path = (
    PROJECT_ROOT
    / "app"
    / "btp_service.py"
)

service_path.write_text(
    service_source.strip() + "\n",
    encoding="utf-8",
)

# Ensure app is recognized as a Python package.
init_path = (
    PROJECT_ROOT
    / "app"
    / "__init__.py"
)

init_path.touch(exist_ok=True)

print("BTP-compatible API created:")
print(service_path)

BTP-compatible API created:
/var/www/filebrowser/.projects/b1f2f927-84af-459c-80e3-1fe60dbc8858/sap-finance-ai-assistant/app/btp_service.py


# Create the Cloud Foundry package

In [23]:
# DAY 24 — CLOUD FOUNDRY DEPLOYMENT FILES
#
# Purpose:
# Generate the buildpack, runtime, start-command and dependency files
# required for a future SAP BTP Cloud Foundry deployment.
#
# No credentials or service keys are written to these files.

deployment_directory = (
    PROJECT_ROOT / "deployment"
)

deployment_directory.mkdir(
    parents=True,
    exist_ok=True,
)


manifest_text = """---
applications:
  - name: governed-sap-finance-assistant
    path: ..
    memory: 512M
    disk-quota: 1G
    instances: 1
    random-route: true
    buildpacks:
      - python_buildpack
    command: python -m uvicorn app.btp_service:app --host 0.0.0.0 --port $PORT
    health-check-type: http
    health-check-http-endpoint: /health
    env:
      APP_ENV: production
      LOG_LEVEL: INFO
      AICORE_RESOURCE_GROUP: default
"""

manifest_path = (
    deployment_directory / "manifest.yml"
)

manifest_path.write_text(
    manifest_text,
    encoding="utf-8",
)


runtime_path = (
    PROJECT_ROOT / "runtime.txt"
)

runtime_path.write_text(
    "python-3.11.x\n",
    encoding="utf-8",
)


procfile_path = (
    PROJECT_ROOT / "Procfile"
)

procfile_path.write_text(
    (
        "web: python -m uvicorn app.btp_service:app "
        "--host 0.0.0.0 --port $PORT\n"
    ),
    encoding="utf-8",
)


# Preserve any existing dependencies and add only missing packages.
requirements_path = (
    PROJECT_ROOT / "requirements.txt"
)

if requirements_path.exists():
    requirement_lines = (
        requirements_path.read_text(
            encoding="utf-8"
        ).splitlines()
    )
else:
    requirement_lines = []

required_packages = {
    "fastapi": "fastapi>=0.115,<1.0",
    "uvicorn": "uvicorn[standard]>=0.30,<1.0",
    "pydantic": "pydantic>=2.8,<3.0",
    "httpx": "httpx>=0.27,<1.0",
}

for package_name, package_specification in required_packages.items():
    package_already_present = any(
        line.strip().lower().startswith(
            package_name
        )
        for line in requirement_lines
        if line.strip()
        and not line.strip().startswith("#")
    )

    if not package_already_present:
        requirement_lines.append(
            package_specification
        )

requirements_path.write_text(
    "\n".join(requirement_lines).strip()
    + "\n",
    encoding="utf-8",
)


# Exclude local files that should not be uploaded during deployment.
cfignore_path = (
    PROJECT_ROOT / ".cfignore"
)

ignore_entries = {
    ".git/",
    ".ipynb_checkpoints/",
    "__pycache__/",
    "*.pyc",
    ".env",
    "*.env",
}

if cfignore_path.exists():
    existing_ignore_entries = set(
        cfignore_path.read_text(
            encoding="utf-8"
        ).splitlines()
    )
else:
    existing_ignore_entries = set()

combined_ignore_entries = sorted(
    existing_ignore_entries
    | ignore_entries
)

cfignore_path.write_text(
    "\n".join(combined_ignore_entries)
    + "\n",
    encoding="utf-8",
)


print("Deployment package created:\n")

for created_path in [
    manifest_path,
    runtime_path,
    procfile_path,
    requirements_path,
    cfignore_path,
]:
    print(created_path)

Deployment package created:

/var/www/filebrowser/.projects/b1f2f927-84af-459c-80e3-1fe60dbc8858/sap-finance-ai-assistant/deployment/manifest.yml
/var/www/filebrowser/.projects/b1f2f927-84af-459c-80e3-1fe60dbc8858/sap-finance-ai-assistant/runtime.txt
/var/www/filebrowser/.projects/b1f2f927-84af-459c-80e3-1fe60dbc8858/sap-finance-ai-assistant/Procfile
/var/www/filebrowser/.projects/b1f2f927-84af-459c-80e3-1fe60dbc8858/sap-finance-ai-assistant/requirements.txt
/var/www/filebrowser/.projects/b1f2f927-84af-459c-80e3-1fe60dbc8858/sap-finance-ai-assistant/.cfignore


# Run API smoke tests

In [24]:
# DAY 24 — LOCAL API SMOKE TESTS
#
# Purpose:
# Test health, readiness, grounded answers, human escalation,
# production blocking and audit logging without starting a server.

from fastapi.testclient import TestClient


# Test normal behavior in the local development environment.
original_app_environment = os.environ.get(
    "APP_ENV"
)

os.environ["APP_ENV"] = "development"


module_spec = importlib.util.spec_from_file_location(
    "day24_btp_service",
    service_path,
)

service_module = (
    importlib.util.module_from_spec(
        module_spec
    )
)

module_spec.loader.exec_module(
    service_module
)

client = TestClient(
    service_module.app
)


health_response = client.get(
    "/health"
)

ready_response = client.get(
    "/ready"
)

configuration_response = client.get(
    "/configuration"
)


grounded_response = client.post(
    "/ask",
    json={
        "user_id": "synthetic-user-001",
        "role": "finance_analyst",
        "business_unit": "NA",
        "question": (
            "What approval is required for a "
            "USD 125,000 journal entry?"
        ),
    },
)


escalation_response = client.post(
    "/ask",
    json={
        "user_id": "synthetic-user-002",
        "role": "finance_analyst",
        "business_unit": "NA",
        "question": (
            "Post a USD 125,000 journal entry now."
        ),
    },
)


# Confirm that an unauthorized production request is blocked.
os.environ["APP_ENV"] = "production"

production_response = client.post(
    "/ask",
    json={
        "user_id": "synthetic-user-003",
        "role": "finance_manager",
        "business_unit": "NA",
        "question": (
            "What approval is required for a journal entry?"
        ),
    },
)


# Restore the environment to its original condition.
if original_app_environment is None:
    os.environ.pop(
        "APP_ENV",
        None,
    )
else:
    os.environ[
        "APP_ENV"
    ] = original_app_environment


grounded_payload = (
    grounded_response.json()
    if grounded_response.status_code == 200
    else {}
)

escalation_payload = (
    escalation_response.json()
    if escalation_response.status_code == 200
    else {}
)


smoke_results = {
    "health_endpoint": (
        health_response.status_code == 200
        and health_response.json().get(
            "status"
        )
        == "healthy"
    ),
    "technical_readiness_endpoint": (
        ready_response.status_code == 200
        and ready_response.json().get(
            "technical_gate"
        )
        == "PASS"
    ),
    "grounded_answer": (
        grounded_response.status_code == 200
        and grounded_payload[
            "result"
        ]["decision"]
        == "answer"
        and grounded_payload[
            "result"
        ]["citations"]
        == ["FIN-JE-001"]
    ),
    "human_approval_escalation": (
        escalation_response.status_code == 200
        and escalation_payload[
            "result"
        ]["decision"]
        == "escalate"
        and escalation_payload[
            "result"
        ][
            "requires_human_approval"
        ]
        is True
    ),
    "unauthorized_production_block": (
        production_response.status_code == 503
    ),
    "audit_log_created": (
        PROJECT_ROOT
        / "audit"
        / "day24_api_audit.jsonl"
    ).exists(),
}


smoke_test_report = {
    "test_run": "day24-btp-api-smoke-test",
    "generated_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "results": smoke_results,
    "http_statuses": {
        "health": health_response.status_code,
        "ready": ready_response.status_code,
        "grounded_answer": grounded_response.status_code,
        "escalation": escalation_response.status_code,
        "production_request": production_response.status_code,
    },
}


smoke_report_path = save_json(
    "artifacts/day24_btp_smoke_test.json",
    smoke_test_report,
)


print("Day 24 smoke tests:\n")

for test_name, test_passed in smoke_results.items():
    icon = "✅" if test_passed else "❌"
    print(f"{icon} {test_name}")

print("\nHTTP statuses:")
print(
    smoke_test_report[
        "http_statuses"
    ]
)

print("\nSmoke-test report:")
print(smoke_report_path)

Day 24 smoke tests:

✅ health_endpoint
✅ technical_readiness_endpoint
✅ grounded_answer
✅ human_approval_escalation
✅ unauthorized_production_block
✅ audit_log_created

HTTP statuses:
{'health': 200, 'ready': 200, 'grounded_answer': 200, 'escalation': 200, 'production_request': 503}

Smoke-test report:
/var/www/filebrowser/.projects/b1f2f927-84af-459c-80e3-1fe60dbc8858/sap-finance-ai-assistant/artifacts/day24_btp_smoke_test.json


# Create the deployment-readiness record and notes

In [25]:
# DAY 24 — DEPLOYMENT READINESS AND DOCUMENTATION
#
# Purpose:
# Scan today's generated files for common credential patterns,
# create a deployment-readiness report and document the architecture.

files_to_scan = [
    service_path,
    manifest_path,
    runtime_path,
    procfile_path,
    requirements_path,
]

secret_patterns = [
    "github_pat_",
    "ghp_",
    "Bearer eyJ",
    "AICORE_CLIENT_SECRET=",
    "sk-",
]

secret_findings = []

for scanned_file in files_to_scan:
    file_contents = scanned_file.read_text(
        encoding="utf-8"
    )

    for secret_pattern in secret_patterns:
        if secret_pattern in file_contents:
            secret_findings.append(
                {
                    "file": str(
                        scanned_file.relative_to(
                            PROJECT_ROOT
                        )
                    ),
                    "pattern": secret_pattern,
                }
            )


deployment_files = [
    "app/btp_service.py",
    "deployment/manifest.yml",
    "runtime.txt",
    "Procfile",
    "requirements.txt",
    ".cfignore",
]

deployment_readiness = {
    "assessment_id": "day24-btp-deployment-readiness",
    "generated_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "deployment_files": deployment_files,
    "all_deployment_files_exist": all(
        (PROJECT_ROOT / relative_path).exists()
        for relative_path in deployment_files
    ),
    "smoke_tests": smoke_results,
    "secret_scan_clean": (
        len(secret_findings) == 0
    ),
    "secret_findings": secret_findings,
    "sap_ai_core_configuration": (
        configuration_response.json()
    ),
    "technical_gate": day23_decision[
        "automated_technical_gate"
    ],
    "human_approval_status": day23_decision[
        "human_approval_status"
    ],
    "deployment_authorized": day23_decision[
        "deployment_authorized"
    ],
    "next_deployment_requirements": [
        "SAP BTP Cloud Foundry subaccount and space",
        "Bound SAP AI Core service instance",
        "Bound authentication service such as XSUAA",
        "Human deployment authorization",
        "Production monitoring and alert configuration",
    ],
}


readiness_path = save_json(
    "artifacts/day24_deployment_readiness.json",
    deployment_readiness,
)


notes_text = f"""# Day 24 — SAP BTP Integration and Deployment

## Objective

Package the governed SAP Finance assistant as a Python HTTP API
suitable for future SAP BTP Cloud Foundry deployment.

## API endpoints

- `/health` confirms that the application process is operating.
- `/ready` confirms that the automated technical gate passed.
- `/configuration` reports configuration presence without exposing values.
- `/ask` returns governed Finance decisions and creates audit events.

## Governance behavior

- The technical gate is `{day23_decision["automated_technical_gate"]}`.
- Human approval is `{day23_decision["human_approval_status"]}`.
- Deployment authorization is `{day23_decision["deployment_authorized"]}`.
- Production requests are blocked until deployment authorization is true.
- Transaction-execution requests are escalated for human approval.
- Audit logging stores a question hash instead of the raw question.

## SAP BTP design

Cloud Foundry will use the Python buildpack, the platform-provided
PORT and the `/health` endpoint. SAP AI Core credentials will be
provided through environment variables or bound services rather
than committed to GitHub.

## Current limitation

This version uses deterministic synthetic responses. It does not
yet call SAP Generative AI Hub. The next integration step is to
replace the deterministic response generator with the SAP Cloud
SDK for AI while preserving the same governance contract.

## Interview explanation

I separated application health, technical readiness and deployment
authorization. This prevents a technically functioning AI service
from receiving production Finance traffic before governance
approval is recorded.
"""


notes_path = (
    PROJECT_ROOT
    / "notes"
    / "day24_btp_integration_deployment.md"
)

notes_path.write_text(
    notes_text,
    encoding="utf-8",
)


print("Deployment readiness report:")
print(readiness_path)

print("\nEngineering notes:")
print(notes_path)

print("\nSecret scan clean:")
print(
    deployment_readiness[
        "secret_scan_clean"
    ]
)

Deployment readiness report:
/var/www/filebrowser/.projects/b1f2f927-84af-459c-80e3-1fe60dbc8858/sap-finance-ai-assistant/artifacts/day24_deployment_readiness.json

Engineering notes:
/var/www/filebrowser/.projects/b1f2f927-84af-459c-80e3-1fe60dbc8858/sap-finance-ai-assistant/notes/day24_btp_integration_deployment.md

Secret scan clean:
False


# One completion check

In [26]:
# DAY 24 — CORRECTED SECRET SCAN

from pathlib import Path
import re


secret_patterns = {
    "GitHub token": re.compile(
        r"\b(?:gh[pousr]_[A-Za-z0-9]{20,}"
        r"|github_pat_[A-Za-z0-9_]{20,})\b"
    ),
    "AWS access key": re.compile(
        r"\bAKIA[A-Z0-9]{16}\b"
    ),
    "Private key": re.compile(
        r"-----BEGIN (?:RSA |EC |OPENSSH )?"
        r"PRIVATE KEY-----"
    ),
    "Credential inside URL": re.compile(
        r"https://[^/\s:@]+:[^@\s/]+@"
    ),
    "Hard-coded credential": re.compile(
        r"""(?imx)
        ^\s*
        ["']?
        (?:
            api[_-]?key
            |access[_-]?token
            |auth[_-]?token
            |github[_-]?token
            |password
            |client[_-]?secret
        )
        ["']?
        \s*[:=]\s*
        ["']?
        (?P<value>[^ \t\r\n"'#,}\]]+)
        """
    ),
}


safe_runtime_values = (
    "${",
    "((",
    "<",
    "os.getenv",
    "os.environ",
    "getpass",
)


secret_findings = []


for scanned_file in files_to_scan:
    scanned_file = Path(scanned_file)

    if not scanned_file.is_absolute():
        scanned_file = (
            PROJECT_ROOT
            / scanned_file
        )

    if not scanned_file.exists():
        continue

    file_text = scanned_file.read_text(
        encoding="utf-8"
    )

    for rule_name, pattern in (
        secret_patterns.items()
    ):
        for match in pattern.finditer(
            file_text
        ):
            if rule_name == "Hard-coded credential":
                detected_value = (
                    match.group("value")
                    .strip()
                )

                if (
                    not detected_value
                    or detected_value.startswith(
                        safe_runtime_values
                    )
                    or detected_value.lower()
                    in {
                        "none",
                        "null",
                        "placeholder",
                        "replace-me",
                        "replace_me",
                    }
                ):
                    continue

            line_number = (
                file_text.count(
                    "\n",
                    0,
                    match.start(),
                )
                + 1
            )

            secret_findings.append(
                {
                    "file": str(
                        scanned_file.relative_to(
                            PROJECT_ROOT
                        )
                    ),
                    "line": line_number,
                    "rule": rule_name,
                }
            )


if secret_findings:
    print("❌ Potential secrets detected:")

    for finding in secret_findings:
        print(
            finding["file"],
            "line",
            finding["line"],
            "—",
            finding["rule"],
            "— <value redacted>",
        )
else:
    print("✅ Secret scan is clean.")

✅ Secret scan is clean.


In [9]:
# LIST SECRET-SCAN VARIABLES SAFELY

related_names = [
    name
    for name in globals()
    if any(
        word in name.lower()
        for word in [
            "secret",
            "credential",
            "token",
            "scan",
        ]
    )
]

print("Security-related variables:")

for name in sorted(related_names):
    value = globals()[name]

    if isinstance(
        value,
        (bool, int, float, type(None)),
    ):
        print(
            name,
            "=",
            value,
        )

    elif isinstance(value, (list, tuple, set)):
        print(
            name,
            f"— {type(value).__name__}, "
            f"{len(value)} item(s)"
        )

    elif isinstance(value, dict):
        print(
            name,
            "— dictionary keys:",
            list(value.keys()),
        )

    else:
        print(
            name,
            "—",
            type(value).__name__,
            "(value hidden)",
        )

Security-related variables:
files_to_scan — list, 5 item(s)
scanned_file — PosixPath (value hidden)
secret_findings — list, 0 item(s)
secret_pattern — str (value hidden)
secret_patterns — dictionary keys: ['GitHub token', 'AWS access key', 'Private key', 'Credential inside URL', 'Hard-coded credential']


In [30]:
# DAY 24 — CORRECTED SECRET SCAN

from pathlib import Path
import re


secret_patterns = {
    "GitHub token": re.compile(
        r"\b(?:gh[pousr]_[A-Za-z0-9]{20,}"
        r"|github_pat_[A-Za-z0-9_]{20,})\b"
    ),
    "AWS access key": re.compile(
        r"\bAKIA[A-Z0-9]{16}\b"
    ),
    "Private key": re.compile(
        r"-----BEGIN (?:RSA |EC |OPENSSH )?"
        r"PRIVATE KEY-----"
    ),
    "Credential inside URL": re.compile(
        r"https://[^/\s:@]+:[^@\s/]+@"
    ),
    "Hard-coded credential": re.compile(
        r"""(?imx)
        ^\s*
        ["']?
        (?:
            api[_-]?key
            |access[_-]?token
            |auth[_-]?token
            |github[_-]?token
            |password
            |client[_-]?secret
        )
        ["']?
        \s*[:=]\s*
        ["']?
        (?P<value>[^ \t\r\n"'#,}\]]+)
        """
    ),
}


safe_runtime_values = (
    "${",
    "((",
    "<",
    "os.getenv",
    "os.environ",
    "getpass",
)


secret_findings = []


for scanned_file in files_to_scan:
    scanned_file = Path(scanned_file)

    if not scanned_file.is_absolute():
        scanned_file = (
            PROJECT_ROOT
            / scanned_file
        )

    if not scanned_file.exists():
        continue

    file_text = scanned_file.read_text(
        encoding="utf-8"
    )

    for rule_name, pattern in (
        secret_patterns.items()
    ):
        for match in pattern.finditer(
            file_text
        ):
            if rule_name == "Hard-coded credential":
                detected_value = (
                    match.group("value")
                    .strip()
                )

                if (
                    not detected_value
                    or detected_value.startswith(
                        safe_runtime_values
                    )
                    or detected_value.lower()
                    in {
                        "none",
                        "null",
                        "placeholder",
                        "replace-me",
                        "replace_me",
                    }
                ):
                    continue

            line_number = (
                file_text.count(
                    "\n",
                    0,
                    match.start(),
                )
                + 1
            )

            secret_findings.append(
                {
                    "file": str(
                        scanned_file.relative_to(
                            PROJECT_ROOT
                        )
                    ),
                    "line": line_number,
                    "rule": rule_name,
                }
            )


if secret_findings:
    print("❌ Potential secrets detected:")

    for finding in secret_findings:
        print(
            finding["file"],
            "line",
            finding["line"],
            "—",
            finding["rule"],
            "— <value redacted>",
        )
else:
    print("✅ Secret scan is clean.")

✅ Secret scan is clean.


In [34]:
# DAY 24 — REPAIR NOTEBOOK PATH AND RECHECK SMOKE REPORT

from pathlib import Path
import json
import shutil


# ---------------------------------------------------------
# 1. COPY THE SAVED NOTEBOOK INTO notebooks/
# ---------------------------------------------------------

notebook_name = (
    "day24_btp_api_packaging.ipynb"
)

notebook_destination = (
    PROJECT_ROOT
    / "notebooks"
    / notebook_name
)

notebook_candidates = [
    PROJECT_ROOT / notebook_name,
    *[
        path
        for path in PROJECT_ROOT.rglob(
            "*day24*.ipynb"
        )
        if ".ipynb_checkpoints"
        not in path.parts
        and path != notebook_destination
    ],
]

notebook_source = next(
    (
        path
        for path in notebook_candidates
        if path.exists()
    ),
    None,
)

if notebook_source is None:
    raise FileNotFoundError(
        "Save the open notebook as "
        f"{notebook_name}, then rerun this cell."
    )

notebook_destination.parent.mkdir(
    parents=True,
    exist_ok=True,
)

shutil.copy2(
    notebook_source,
    notebook_destination,
)

print(
    "✅ Notebook copied to:",
    notebook_destination.relative_to(
        PROJECT_ROOT
    ),
)


# ---------------------------------------------------------
# 2. RELOAD THE SMOKE-TEST REPORT
# ---------------------------------------------------------

smoke_report_path = (
    PROJECT_ROOT
    / "artifacts"
    / "day24_btp_smoke_test.json"
)

smoke_report = json.loads(
    smoke_report_path.read_text(
        encoding="utf-8"
    )
)


def result_passed(value):
    if isinstance(value, bool):
        return value

    if isinstance(value, dict):
        for key in [
            "passed",
            "success",
            "ok",
        ]:
            if key in value:
                return result_passed(
                    value[key]
                )

        for key in [
            "result",
            "status",
            "outcome",
        ]:
            if key in value:
                return result_passed(
                    value[key]
                )

    if isinstance(value, str):
        return value.strip().upper() in {
            "PASS",
            "PASSED",
            "SUCCESS",
            "TRUE",
            "OK",
        }

    return False


def find_check(node, target_name):
    if isinstance(node, dict):
        if target_name in node:
            return node[target_name]

        for identity_key in [
            "name",
            "check",
            "test",
            "id",
        ]:
            if (
                str(
                    node.get(identity_key, "")
                )
                == target_name
            ):
                return node

        for child in node.values():
            result = find_check(
                child,
                target_name,
            )

            if result is not None:
                return result

    elif isinstance(node, list):
        for child in node:
            result = find_check(
                child,
                target_name,
            )

            if result is not None:
                return result

    return None


def extract_http_status(value):
    if (
        isinstance(value, int)
        and not isinstance(value, bool)
    ):
        return value

    if isinstance(value, dict):
        for key in [
            "status_code",
            "http_status",
            "status",
        ]:
            candidate = value.get(key)

            if (
                isinstance(candidate, int)
                and not isinstance(
                    candidate,
                    bool,
                )
            ):
                return candidate

    return None


def find_http_status(node, target_name):
    if isinstance(node, dict):
        if target_name in node:
            status = extract_http_status(
                node[target_name]
            )

            if status is not None:
                return status

        for identity_key in [
            "name",
            "request",
            "endpoint",
            "id",
        ]:
            if (
                str(
                    node.get(identity_key, "")
                )
                == target_name
            ):
                status = extract_http_status(
                    node
                )

                if status is not None:
                    return status

        for child in node.values():
            status = find_http_status(
                child,
                target_name,
            )

            if status is not None:
                return status

    elif isinstance(node, list):
        for child in node:
            status = find_http_status(
                child,
                target_name,
            )

            if status is not None:
                return status

    return None


required_smoke_checks = [
    "health_endpoint",
    "technical_readiness_endpoint",
    "grounded_answer",
    "human_approval_escalation",
    "unauthorized_production_block",
    "audit_log_created",
]


check_results = {
    check_name: result_passed(
        find_check(
            smoke_report,
            check_name,
        )
    )
    for check_name in required_smoke_checks
}


print("\nSmoke-test results from saved report:")

for check_name, passed in (
    check_results.items()
):
    print(
        "✅" if passed else "❌",
        check_name,
    )


production_status = find_http_status(
    smoke_report,
    "production_request",
)

production_blocked = (
    check_results[
        "unauthorized_production_block"
    ]
    and production_status
    in {
        401,
        403,
        423,
        503,
    }
)


print(
    "\nProduction request HTTP status:",
    production_status,
)


# ---------------------------------------------------------
# 3. UPDATE AND RERUN THE COMPLETION CONDITIONS
# ---------------------------------------------------------

missing_day24_files = [
    str(relative_path)
    for relative_path in required_day24_files
    if not (
        PROJECT_ROOT
        / relative_path
    ).exists()
]

completion_conditions[
    "all Day 24 files exist"
] = (
    len(missing_day24_files) == 0
)

completion_conditions[
    "all API smoke tests passed"
] = all(
    check_results.values()
)

completion_conditions[
    "production request was blocked"
] = production_blocked

completion_conditions[
    "secret scan is clean"
] = (
    len(secret_findings) == 0
)


print("\nDay 24 completion check:\n")

for condition_name, passed in (
    completion_conditions.items()
):
    print(
        "✅" if passed else "❌",
        condition_name,
    )


failed_conditions = [
    name
    for name, passed
    in completion_conditions.items()
    if not passed
]


if failed_conditions:
    raise AssertionError(
        "Day 24 is not complete: "
        f"{failed_conditions}"
    )


print(
    "\n✅ DAY 24 COMPLETE — "
    "deployment remains safely unauthorized."
)

FileNotFoundError: Save the open notebook as day24_btp_api_packaging.ipynb, then rerun this cell.

In [31]:
# DAY 24 — RESTORED FINAL COMPLETION CHECK

from pathlib import Path
import json


# ---------------------------------------------------------
# 1. LOCATE THE GIT REPOSITORY
# ---------------------------------------------------------

current_folder = Path.cwd().resolve()

repository_candidates = [
    current_folder,
    *current_folder.parents,
]

projects_area = Path(
    "/var/www/filebrowser/.projects"
)

if projects_area.exists():
    repository_candidates.extend(
        projects_area.glob(
            "*/sap-finance-ai-assistant"
        )
    )


PROJECT_ROOT = next(
    (
        path.resolve()
        for path in repository_candidates
        if path.exists()
        and (path / ".git").exists()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "The Git repository could not be located."
    )


# ---------------------------------------------------------
# 2. VERIFY THE DAY 24 DELIVERABLES
# ---------------------------------------------------------

required_day24_files = [
    Path("app") / "btp_service.py",
    Path("deployment") / "manifest.yml",
    Path("runtime.txt"),
    Path("Procfile"),
    Path("requirements.txt"),
    Path(".cfignore"),
    (
        Path("artifacts")
        / "day24_btp_smoke_test.json"
    ),
    (
        Path("artifacts")
        / "day24_deployment_readiness.json"
    ),
    (
        Path("audit")
        / "day24_api_audit.jsonl"
    ),
    (
        Path("notes")
        / "day24_btp_integration_deployment.md"
    ),
    (
        Path("notebooks")
        / "day24_btp_api_packaging.ipynb"
    ),
]


missing_day24_files = [
    str(relative_path)
    for relative_path in required_day24_files
    if not (
        PROJECT_ROOT
        / relative_path
    ).exists()
]


# ---------------------------------------------------------
# 3. LOAD THE SAVED REPORTS
# ---------------------------------------------------------

smoke_report_path = (
    PROJECT_ROOT
    / "artifacts"
    / "day24_btp_smoke_test.json"
)

readiness_report_path = (
    PROJECT_ROOT
    / "artifacts"
    / "day24_deployment_readiness.json"
)


if not smoke_report_path.exists():
    raise FileNotFoundError(
        smoke_report_path
    )

if not readiness_report_path.exists():
    raise FileNotFoundError(
        readiness_report_path
    )


smoke_report = json.loads(
    smoke_report_path.read_text(
        encoding="utf-8"
    )
)

readiness_report = json.loads(
    readiness_report_path.read_text(
        encoding="utf-8"
    )
)


# ---------------------------------------------------------
# 4. INTERPRET BOOLEAN AND STATUS VALUES
# ---------------------------------------------------------

def condition_passed(value):
    if isinstance(value, bool):
        return value

    if isinstance(value, dict):
        for key in [
            "passed",
            "success",
            "ok",
        ]:
            if key in value:
                return condition_passed(
                    value[key]
                )

        return False

    if isinstance(value, str):
        return value.strip().upper() in {
            "PASS",
            "PASSED",
            "TRUE",
            "READY",
            "APPROVED",
            "APPROVED_FOR_TECHNICAL_TESTING",
            "APPROVED_FOR_TECHNICAL_DEPLOYMENT_TESTING",
        }

    return False


smoke_checks = smoke_report.get(
    "checks",
    {},
)

http_statuses = smoke_report.get(
    "http_statuses",
    {},
)


required_smoke_checks = [
    "health_endpoint",
    "technical_readiness_endpoint",
    "grounded_answer",
    "human_approval_escalation",
    "unauthorized_production_block",
    "audit_log_created",
]


all_smoke_tests_passed = (
    all(
        check_name in smoke_checks
        and condition_passed(
            smoke_checks[check_name]
        )
        for check_name in required_smoke_checks
    )
)


production_request_blocked = (
    http_statuses.get(
        "production_request"
    )
    in {
        401,
        403,
        423,
        503,
    }
    and condition_passed(
        smoke_checks.get(
            "unauthorized_production_block",
            False,
        )
    )
)


technical_gate_passed = condition_passed(
    readiness_report.get(
        "technical_gate",
        False,
    )
)


human_approval_status = str(
    readiness_report.get(
        "human_approval_status",
        "",
    )
).strip().upper()


human_approval_pending = (
    "PENDING"
    in human_approval_status
)


deployment_authorized = (
    readiness_report.get(
        "deployment_authorized"
    )
)


deployment_unauthorized = (
    deployment_authorized is False
    or str(
        deployment_authorized
    ).strip().upper()
    in {
        "FALSE",
        "NO",
        "UNAUTHORIZED",
        "NOT_AUTHORIZED",
    }
)


if "secret_findings" not in globals():
    raise NameError(
        "Run the corrected Day 24 secret "
        "scanner before this completion check."
    )


secret_scan_clean = (
    len(secret_findings) == 0
)


# ---------------------------------------------------------
# 5. FINAL COMPLETION CONDITIONS
# ---------------------------------------------------------

completion_conditions = {
    "all Day 24 files exist": (
        len(missing_day24_files) == 0
    ),
    "Day 23 technical gate passed": (
        technical_gate_passed
    ),
    "all API smoke tests passed": (
        all_smoke_tests_passed
    ),
    "production request was blocked": (
        production_request_blocked
    ),
    "secret scan is clean": (
        secret_scan_clean
    ),
    "human approval remains pending": (
        human_approval_pending
    ),
    "deployment remains unauthorized": (
        deployment_unauthorized
    ),
}


print("Day 24 completion check:\n")

for condition_name, passed in (
    completion_conditions.items()
):
    icon = "✅" if passed else "❌"

    print(
        icon,
        condition_name,
    )


if missing_day24_files:
    print(
        "\nMissing files:",
        missing_day24_files,
    )


failed_conditions = [
    condition_name
    for condition_name, passed
    in completion_conditions.items()
    if not passed
]


if failed_conditions:
    raise AssertionError(
        "Day 24 is not complete: "
        f"{failed_conditions}"
    )


print(
    "\n✅ DAY 24 COMPLETE — "
    "deployment remains safely unauthorized."
)

Day 24 completion check:

❌ all Day 24 files exist
✅ Day 23 technical gate passed
❌ all API smoke tests passed
❌ production request was blocked
✅ secret scan is clean
✅ human approval remains pending
✅ deployment remains unauthorized

Missing files: ['notebooks/day24_btp_api_packaging.ipynb']


AssertionError: Day 24 is not complete: ['all Day 24 files exist', 'all API smoke tests passed', 'production request was blocked']

In [27]:
# DAY 24 — CORRECTED FINAL COMPLETION CHECK

# Use the result from the corrected secret scanner.
completion_conditions[
    "secret scan is clean"
] = (
    len(secret_findings) == 0
)


print("Day 24 corrected completion check:\n")

for condition_name, condition_passed in (
    completion_conditions.items()
):
    icon = (
        "✅"
        if condition_passed
        else "❌"
    )

    print(
        icon,
        condition_name,
    )


failed_conditions = [
    condition_name
    for condition_name, condition_passed
    in completion_conditions.items()
    if not condition_passed
]


if failed_conditions:
    raise AssertionError(
        "Day 24 is not complete: "
        f"{failed_conditions}"
    )


print(
    "\n✅ DAY 24 COMPLETE — "
    "deployment remains safely unauthorized."
)

NameError: name 'completion_conditions' is not defined

In [ ]:
# SHOW THE EXACT FAILED DAY 24 CONDITIONS

failed_conditions = [
    condition_name
    for condition_name, condition_passed
    in completion_conditions.items()
    if not condition_passed
]

print("Failed Day 24 conditions:")

if failed_conditions:
    for condition_name in failed_conditions:
        print("❌", condition_name)
else:
    print("✅ No failed conditions.")

In [29]:
# SEARCH NOTEBOOK CHECKPOINTS FOR THE DELETED COMPLETION CELL

from pathlib import Path
import json


search_roots = [
    Path.cwd().resolve(),
]

projects_area = Path(
    "/var/www/filebrowser/.projects"
)

if projects_area.exists():
    search_roots.extend(
        projects_area.glob(
            "*/sap-finance-ai-assistant"
        )
    )


notebook_files = []

for search_root in search_roots:
    if search_root.exists():
        notebook_files.extend(
            search_root.rglob("*.ipynb")
        )


notebook_files = list(
    dict.fromkeys(
        path.resolve()
        for path in notebook_files
    )
)


recovered_cells = []

for notebook_path in notebook_files:
    try:
        notebook_data = json.loads(
            notebook_path.read_text(
                encoding="utf-8"
            )
        )
    except (
        json.JSONDecodeError,
        UnicodeDecodeError,
        OSError,
    ):
        continue

    for cell in notebook_data.get(
        "cells",
        [],
    ):
        source = "".join(
            cell.get("source", [])
        )

        if (
            "completion_conditions"
            in source
            and "all Day 24 files exist"
            in source
        ):
            recovered_cells.append(
                (
                    notebook_path,
                    source,
                )
            )


if recovered_cells:
    for number, (
        notebook_path,
        source,
    ) in enumerate(
        recovered_cells,
        start=1,
    ):
        print(
            f"\n===== RECOVERED CELL {number} ====="
        )
        print("Source:", notebook_path)
        print()
        print(source)

else:
    print(
        "The original Day 24 completion cell "
        "was not found in any saved notebook "
        "or checkpoint."
    )

The original Day 24 completion cell was not found in any saved notebook or checkpoint.
